# Movie Auto-Dub

Free and open source: no paid APIs, no login or token, no restrictive licenses.

Demucs (split vocals from music) -> faster-whisper (transcribe) -> Resemblyzer (figure out who's
speaking) -> Argos Translate -> Chatterbox (clone each voice, speak the translation) -> reassemble
on the original timeline.

Before running:
- `Runtime > Change runtime type > T4 GPU`
- Have an audio file ready (not video - just the audio track)
- Set `NUM_SPEAKERS` and the language codes in the config cell
- Try a short clip first

## Setup

In [ ]:
import subprocess

# Checked via nvidia-smi (not torch) so this doesn't import torch before the pinned
# torch==2.6.0 install below - otherwise this session's torch stays on Colab's
# preinstalled build even after that install overwrites the on-disk files.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True)
assert gpu.returncode == 0, "Turn on a GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", gpu.stdout.strip())

GPU: Tesla T4


In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null

# The "| grep -Ev ..." hides pip's dependency-conflict notices, which come from unrelated
# packages already in Colab's environment (fastai, gradio, etc. - not used here). torchvision
# used to be in that category too, but it turns out chatterbox's transformers dependency does
# import it internally, so it's pinned properly below rather than left alone or removed.
# "set -o pipefail" keeps a REAL pip failure visible despite the pipe - this only hides that one
# specific harmless message pattern, nothing else.
!set -o pipefail; pip install -q demucs faster-whisper chatterbox-tts argostranslate resemblyzer scikit-learn pydub librosa soundfile pyloudnorm "numpy>=2.0,<2.6" 2>&1 | grep -Ev "dependency resolver does not currently|but you have .* which is incompatible|which is not installed\.$"

# Chatterbox needs exactly this version - pin it explicitly rather than trusting the resolver.
!set -o pipefail; pip install -q torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 2>&1 | grep -Ev "dependency resolver does not currently|but you have .* which is incompatible|which is not installed\.$"

import torch
print()
print("=" * 40)
print("INSTALL OK - torch", torch.__version__, "- CUDA:", torch.cuda.is_available())
print("=" * 40)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 792.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/

In [ ]:
!python -c "from chatterbox.mtl_tts import ChatterboxMultilingualTTS; print('chatterbox ready')"

chatterbox ready


In [ ]:
import os

NUM_SPEAKERS = 2    # how many people talk in the clip
LANG_SRC = "en"      # source language
LANG_TGT = "hi"      # target language - hi, es, fr, de, ja, zh, ar, ru, ...

MAX_LINE_SECONDS = 12   # split a line if it runs longer than this
MERGE_GAP = 0.3         # join same-speaker lines closer together than this

device = "cuda"
WORKDIR = "/content/dub"
os.makedirs(WORKDIR, exist_ok=True)

## Upload audio and separate vocals from music

In [ ]:
import os
import soundfile as sf
from google.colab import files

print("Upload an audio file:")
uploaded = files.upload()
name = list(uploaded.keys())[0]
AUDIO_PATH = f"{WORKDIR}/{name}"
os.rename(name, AUDIO_PATH)

AUDIO_DURATION = sf.info(AUDIO_PATH).duration
print(f"Loaded {name}, {AUDIO_DURATION:.1f}s")

Upload an audio file:


Saving Perfect.mp3 to Perfect.mp3
Loaded Perfect.mp3, 279.9s


In [ ]:
import os

DEMUCS_OUT = f"{WORKDIR}/demucs_out"
!python -m demucs --two-stems=vocals -n htdemucs -o "{DEMUCS_OUT}" "{AUDIO_PATH}"

track_name = os.path.splitext(os.path.basename(AUDIO_PATH))[0]
VOCALS_PATH = f"{DEMUCS_OUT}/htdemucs/{track_name}/vocals.wav"
BACKGROUND_PATH = f"{DEMUCS_OUT}/htdemucs/{track_name}/no_vocals.wav"
assert os.path.exists(VOCALS_PATH), "Demucs output not found - check the log above"

htdemucs.yaml: 100% 21.0/21.0 [00:00<00:00, 106kB/s]

955717e8.safetensors: downloading bytes:  47% 39.5M/84.0M [00:00<00:00, 60.1MB/s,   ???B/s  ]
955717e8.safetensors: downloading bytes:  71% 59.9M/84.0M [00:00<00:00, 87.2MB/s, 3.91MB/s  ]
955717e8.safetensors: downloading bytes: 100% 84.0M/84.0M [00:01<00:00, 76.5MB/s, 8.14MB/s  ]
955717e8.safetensors: reconstructing file: 100% 84.0M/84.0M [00:01<00:00, 76.5MB/s, 8.19MB/s  ]
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /content/dub/demucs_out/htdemucs
Separating track /content/dub/Perfect.mp3
100%|██████████████████████████████████████████████| 280.79999999999995/280.79999999999995 [00:16<00:00, 17.24seconds/s]


## Transcribe

In [ ]:
import torch, gc
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device=device, compute_type="float16")
raw_segments, _ = model.transcribe(VOCALS_PATH, language=LANG_SRC, word_timestamps=True)

segments = []
for seg in raw_segments:
    words = [{"word": w.word, "start": w.start, "end": w.end} for w in (seg.words or [])]
    if words:
        segments.append({"start": seg.start, "end": seg.end, "text": seg.text.strip(), "words": words})

del model
gc.collect()
torch.cuda.empty_cache()
print(f"{len(segments)} lines transcribed")

101 lines transcribed


## Figure out who's talking

In [ ]:
import torch
import soundfile as sf
from resemblyzer import VoiceEncoder, preprocess_wav
from sklearn.cluster import AgglomerativeClustering

# Resemblyzer's bundled checkpoint predates PyTorch 2.6's safer (but stricter) default for
# torch.load, which trips on this specific old file. This calls the real loader directly
# (torch.serialization.load, a path this patch never modifies) instead of saving "whatever
# torch.load currently is" - that's what broke on a re-run, since torch.load was by then
# already a wrapper from an earlier run, not the original.
def _load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return torch.serialization.load(*args, **kwargs)
torch.load = _load_compat

encoder = VoiceEncoder(device=device)
vocals, vocals_sr = sf.read(VOCALS_PATH)
if vocals.ndim > 1:
    vocals = vocals.mean(axis=1)

def embed(start, end):
    clip = vocals[int(start * vocals_sr):int(end * vocals_sr)]
    return encoder.embed_utterance(preprocess_wav(clip, source_sr=vocals_sr))

embeddings = [embed(s["start"], s["end"]) for s in segments]
labels = AgglomerativeClustering(n_clusters=NUM_SPEAKERS, metric="cosine", linkage="average").fit_predict(embeddings)
for seg, label in zip(segments, labels):
    seg["speaker"] = f"speaker_{label}"

print("Speakers found:", {f"speaker_{i}": int((labels == i).sum()) for i in range(NUM_SPEAKERS)})

Loaded the voice encoder model on cuda in 0.21 seconds.
Speakers found: {'speaker_0': 42, 'speaker_1': 59}


## Turn the transcript into dubbing lines

In [ ]:
def merge_same_speaker(segments):
    merged = []
    for seg in segments:
        words = [w for w in seg["words"] if "start" in w and "end" in w]
        if not words:
            continue
        line = {"speaker": seg["speaker"], "start": words[0]["start"], "end": words[-1]["end"],
                "text": seg["text"], "words": words}
        prev = merged[-1] if merged else None
        if prev and prev["speaker"] == line["speaker"] and line["start"] - prev["end"] < MERGE_GAP:
            prev["end"] = line["end"]
            prev["text"] += " " + line["text"]
            prev["words"] += line["words"]
        else:
            merged.append(line)
    return merged

def split_long_lines(line):
    words = line["words"]
    pieces, start = [], 0
    for i, w in enumerate(words):
        too_long = words[i]["end"] - words[start]["start"] > MAX_LINE_SECONDS
        end_of_sentence = w["word"].strip().endswith((".", "?", "!"))
        if i == len(words) - 1 or (too_long and end_of_sentence):
            chunk = words[start:i + 1]
            pieces.append({**line, "start": chunk[0]["start"], "end": chunk[-1]["end"],
                           "words": chunk, "text": " ".join(w["word"] for w in chunk)})
            start = i + 1
    return pieces

lines = merge_same_speaker(segments)
utterances = [piece for line in lines for piece in split_long_lines(line)]
print(f"{len(utterances)} lines to dub")

63 lines to dub


## Translate

In [ ]:
import argostranslate.package
import argostranslate.translate

argostranslate.package.update_package_index()
packages = argostranslate.package.get_available_packages()
pkg = next(p for p in packages if p.from_code == LANG_SRC and p.to_code == LANG_TGT)
argostranslate.package.install_from_path(pkg.download())

for u in utterances:
    u["translated_text"] = argostranslate.translate.translate(u["text"], LANG_SRC, LANG_TGT)

print(utterances[0]["text"], "->", utterances[0]["translated_text"])

INFO:argostranslate.utils:('Get https://argos-net.com/v1/translate-en_hi-1_1.argosmodel',)
INFO:argostranslate.utils:('Got https://argos-net.com/v1/translate-en_hi-1_1.argosmodel',)
INFO:argostranslate.utils:('get_installed_languages',)
INFO:argostranslate.utils:('paragraphs:', [' I  found  a  love  for  me'])
INFO:argostranslate.utils:('apply_packaged_translation', ' I  found  a  love  for  me')
INFO:argostranslate.utils:('Splitting sentences using SBD Model: (en) StanzaSentencizer',)
INFO:argostranslate.utils:('sentences', ['I  found  a  love  for  me'])
INFO:argostranslate.utils:('tokenized', [['▁I', '▁found', '▁a', '▁love', '▁for', '▁me']])
INFO:argostranslate.utils:('translated_batches', [TranslationResult(hypotheses=[['▁मुझे', '▁प्यार', '▁मिला']], scores=[-3.7594778537750244], attention=[], logits=[])])
INFO:argostranslate.utils:('value_hypotheses:', [('मुझे प्यार मिला', -3.7594778537750244)])
INFO:argostranslate.utils:('translated_paragraphs:', [[('मुझे प्यार मिला', -3.759477853

 I  found  a  love  for  me -> मुझे प्यार मिला


## Grab a voice sample for each speaker

In [ ]:
import soundfile as sf

vocals, vocals_sr = sf.read(VOCALS_PATH)

speaker_refs = {}
for spk in sorted(set(u["speaker"] for u in utterances)):
    longest = max((u for u in utterances if u["speaker"] == spk), key=lambda u: u["end"] - u["start"])
    clip = vocals[int(longest["start"] * vocals_sr):int(longest["end"] * vocals_sr)]
    path = f"{WORKDIR}/ref_{spk}.wav"
    sf.write(path, clip, vocals_sr)
    speaker_refs[spk] = path

print(speaker_refs)

{'speaker_0': '/content/dub/ref_speaker_0.wav', 'speaker_1': '/content/dub/ref_speaker_1.wav'}


## Generate the dubbed speech

This runs in its own subprocess rather than importing Chatterbox directly here - keeps the
notebook's own kernel simple and avoids the torchaudio loading issues from earlier.

In [ ]:
worker_script = """
import sys, json
import soundfile as sf
import librosa
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

with open(sys.argv[1]) as f:
    utterances = json.load(f)

tts = ChatterboxMultilingualTTS.from_pretrained(device="cuda")

def match_duration(y, sr, target):
    ratio = (len(y) / sr) / target
    if abs(ratio - 1) < 0.02:
        return y
    ratio = min(max(ratio, 0.7), 1.4)
    return librosa.effects.time_stretch(y, rate=ratio)

for i, u in enumerate(utterances):
    try:
        wav = tts.generate(u["text"], language_id=u["lang"], audio_prompt_path=u["ref_path"])
        wav = wav.squeeze().cpu().numpy()
        wav = match_duration(wav, tts.sr, u["target_dur"])
        sf.write(u["out_path"], wav, tts.sr)
    except Exception as e:
        print(f"line {i} failed: {e}")
        continue
    if i % 20 == 0:
        print(f"{i}/{len(utterances)}")

print("done")
"""
with open(f"{WORKDIR}/tts_worker.py", "w") as f:
    f.write(worker_script)

In [ ]:
import json

manifest = [
    {
        "text": u["translated_text"],
        "lang": LANG_TGT,
        "ref_path": speaker_refs[u["speaker"]],
        "target_dur": u["end"] - u["start"],
        "out_path": f"{WORKDIR}/line_{i:04d}.wav",
    }
    for i, u in enumerate(utterances)
]
manifest_path = f"{WORKDIR}/manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f)
print(f"{len(manifest)} lines queued")

63 lines queued


In [ ]:
!python "{WORKDIR}/tts_worker.py" "{manifest_path}"

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0% 0/6 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.20G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.20G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.21G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/3.21G [00:00<?, ?B/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/3.21G [00:00<?, ?B/s]

Fetching 6 files:  17% 1/6 [00:00<00:00,  6.28it/s]
Reconstructing (incomplete total...):   0% 1.92M/3.21G [00:00<04:14, 12.6MB/s]

Reconstructing (incomplete total...):   0% 7.80M/3.21G [00:01<10:15, 5.20MB/s, 12.6MB/s  ]

Reconstructing (incomplete total...):  30% 947M/3.21G [00:15<00:37, 60.8MB/s, 66.5MB/s  ]

Reconstructing (incomplete total...):  58% 1.85G/3.21G [00:40<0

In [ ]:
import os
dubbed_paths = [m["out_path"] if os.path.exists(m["out_path"]) else None for m in manifest]
print(f"{sum(p is not None for p in dubbed_paths)}/{len(dubbed_paths)} lines generated")

63/63 lines generated


## Put it back together

In [ ]:
from pydub import AudioSegment
import pyloudnorm as pyln
import soundfile as sf

TARGET_LUFS = -20.0   # consistent, moderate loudness for every dubbed line

background = AudioSegment.from_wav(BACKGROUND_PATH)
sr = background.frame_rate

dubbed_track = AudioSegment.silent(duration=int(AUDIO_DURATION * 1000) + 2000, frame_rate=sr)
for u, path in zip(utterances, dubbed_paths):
    if path is None:
        continue
    data, wav_sr = sf.read(path)
    loudness = pyln.Meter(wav_sr).integrated_loudness(data)
    if loudness > -70:
        data = pyln.normalize.loudness(data, loudness, TARGET_LUFS)
        sf.write(path, data, wav_sr)
    clip = AudioSegment.from_wav(path).set_frame_rate(sr).set_channels(background.channels)
    dubbed_track = dubbed_track.overlay(clip, position=int(u["start"] * 1000))

OUTPUT_PATH = f"{WORKDIR}/dubbed.wav"
background.overlay(dubbed_track).export(OUTPUT_PATH, format="wav")
print("Saved:", OUTPUT_PATH)

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Saved: /content/dub/dubbed.wav


## Download

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>